In [0]:
%run ../00_Setup/01_Config

In [0]:
%run ../Utils/Common_Utils

In [0]:
PIPELINE_NAME = "Silver Products Pipeline"
RUN_ID = generate_run_id()
START_TIME = start_pipeline()

Pipeline Started : 2026-07-19 10:20:06.083348


In [0]:
SOURCE_TABLE = TARGET_TABLE_PRODUCTS
TARGET_TABLE = SILVER_PRODUCTS
QUARANTINE_TABLE = QUARANTINE_PRODUCTS

In [0]:
# Read products from bronze
products_df = spark.read.table(SOURCE_TABLE)

In [0]:
products_df.printSchema()
total_rows = products_df.count()
print(f"Total rows: {total_rows}")

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_length: string (nullable = true)
 |-- product_description_length: string (nullable = true)
 |-- product_photos_qty: string (nullable = true)
 |-- product_weight_g: string (nullable = true)
 |-- product_length_cm: string (nullable = true)
 |-- product_height_cm: string (nullable = true)
 |-- product_width_cm: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- ingestion_date: date (nullable = true)
 |-- pipeline_name: string (nullable = true)
 |-- run_id: string (nullable = true)

Total rows: 3000


In [0]:
display(products_df.limit(10))

product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,ingestion_timestamp,ingestion_date,pipeline_name,run_id
PROD_000001,music,66,512,5,18964,11,25,52,2026-07-15T07:38:18.366Z,2026-07-15,Bronze_Products,3cfdd221-c500-4089-8cfc-ed5711c59956
PROD_000002,food,40,452,4,25798,57,42,49,2026-07-15T07:38:18.366Z,2026-07-15,Bronze_Products,3cfdd221-c500-4089-8cfc-ed5711c59956
PROD_000003,office,69,121,5,24989,45,16,10,2026-07-15T07:38:18.366Z,2026-07-15,Bronze_Products,3cfdd221-c500-4089-8cfc-ed5711c59956
PROD_000004,music,46,717,4,18661,91,8,48,2026-07-15T07:38:18.366Z,2026-07-15,Bronze_Products,3cfdd221-c500-4089-8cfc-ed5711c59956
PROD_000005,toys,29,632,5,26347,79,32,11,2026-07-15T07:38:18.366Z,2026-07-15,Bronze_Products,3cfdd221-c500-4089-8cfc-ed5711c59956
PROD_000006,books,24,845,1,12830,52,31,68,2026-07-15T07:38:18.366Z,2026-07-15,Bronze_Products,3cfdd221-c500-4089-8cfc-ed5711c59956
PROD_000007,sports,34,563,3,12265,66,7,20,2026-07-15T07:38:18.366Z,2026-07-15,Bronze_Products,3cfdd221-c500-4089-8cfc-ed5711c59956
PROD_000008,books,40,247,4,13678,21,19,63,2026-07-15T07:38:18.366Z,2026-07-15,Bronze_Products,3cfdd221-c500-4089-8cfc-ed5711c59956
PROD_000009,garden,55,170,3,19359,80,8,27,2026-07-15T07:38:18.366Z,2026-07-15,Bronze_Products,3cfdd221-c500-4089-8cfc-ed5711c59956
PROD_000010,toys,34,750,1,16878,45,17,71,2026-07-15T07:38:18.366Z,2026-07-15,Bronze_Products,3cfdd221-c500-4089-8cfc-ed5711c59956


In [0]:
# Data Quality Check
primary_key_status = validate_primary_key(
    products_df,
    "product_id"
)

duplicate_rows  = duplicate_summary(
    products_df,
    total_rows
)

null_summary(products_df)

Total Rows : 3000
Distinct Count : 3000
Primary Key Validation Passed — (product_id)
Duplicate Rows : 0


product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,ingestion_timestamp,ingestion_date,pipeline_name,run_id
0,0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
# Cell Standardization and Cleaning 

In [0]:
products_df = (
    products_df
    .withColumn(
        "product_category_name",
        F.trim(F.lower(F.col("product_category_name")))
    )

)

In [0]:
# Standardization & Type Casting

products_df = (
    products_df
    .withColumn(
        "product_name_length",
        F.col("product_name_length").cast("int")
    )
    .withColumn(
        "product_description_length",
        F.col("product_description_length").cast("int")
    )
    .withColumn(
        "product_photos_qty",
        F.col("product_photos_qty").cast("int")
    )
    .withColumn(
        "product_weight_g",
        F.col("product_weight_g").cast("double")
    )
    .withColumn(
        "product_length_cm",
        F.col("product_length_cm").cast("double")
    )
    .withColumn(
        "product_height_cm",
        F.col("product_height_cm").cast("double")
    )
    .withColumn(
        "product_width_cm",
        F.col("product_width_cm").cast("double")
    )
)

In [0]:
products_df.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_length: integer (nullable = true)
 |-- product_description_length: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: double (nullable = true)
 |-- product_length_cm: double (nullable = true)
 |-- product_height_cm: double (nullable = true)
 |-- product_width_cm: double (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- ingestion_date: date (nullable = true)
 |-- pipeline_name: string (nullable = true)
 |-- run_id: string (nullable = true)



In [0]:
# Derived Column - Product Volume
products_df = products_df.withColumn(
    "product_volume_cm3",
    F.col("product_length_cm") * F.col("product_height_cm") * F.col("product_width_cm")
)

In [0]:
# Business Rule Validation

invalid_products = (
    products_df
    .filter(
        (F.col("product_category_name").isNull())
        |
        (F.trim(F.col("product_category_name")) == "")
        |
        (F.col("product_weight_g") <= 0)
        |
        (F.col("product_length_cm") <= 0)
        |
        (F.col("product_height_cm") <= 0)
        |
        (F.col("product_width_cm") <= 0)
        |
        (F.col("product_photos_qty") < 0)

    )
    .withColumn(
        "quarantine_reason",
        F.when(
            F.col("product_category_name").isNull(),
            "Missing Product Category"
        )
        .when(
            F.trim(F.col("product_category_name")) == "",
            "Empty Product Category"
        )
        .when(
            F.col("product_weight_g") <= 0,
            "Invalid Product Weight"
        )
        .when(
            F.col("product_length_cm") <= 0,
            "Invalid Product Length"
        )
        .when(
            F.col("product_height_cm") <= 0,
            "Invalid Product Height"
        )
        .when(
            F.col("product_width_cm") <= 0,
            "Invalid Product Width"
        )
        .when(
            F.col("product_photos_qty") < 0,
            "Invalid Photo Count"
        )
    )
)
print(f"Business Rule Violations : {invalid_products.count()}")

Business Rule Violations : 0


In [0]:
# Seperate valid records 
valid_products = products_df.join(
    invalid_products.select("product_id"),
    on="product_id",
    how="left_anti"
)
print(f"Valid Records : {valid_products.count()}")

Valid Records : 3000


from pyspark.sql.functions import current_timestamp, lit

valid_products = (
    valid_products
    .withColumn("effective_start_date", current_timestamp())
    .withColumn("effective_end_date", lit(None).cast("timestamp"))
    .withColumn("is_current", lit(True))
)

In [0]:
# Write Quarantine Records

(
    invalid_products.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(QUARANTINE_TABLE)
)
print(f"Quarantined Records : {invalid_products.count()}")

Quarantined Records : 0


In [0]:
# Refresh Audit columns 
valid_products = add_audit_columns(
    valid_products,
    PIPELINE_NAME,
    RUN_ID
)

In [0]:
# ============================================================
# Incremental Load (SCD Type 2)
# ============================================================

from delta.tables import DeltaTable

if not spark.catalog.tableExists(TARGET_TABLE):
    # First run — create table with SCD2 columns
    (
        valid_products
        .withColumn("effective_start_date", F.current_date())
        .withColumn("effective_end_date", F.lit(None).cast("date"))
        .withColumn("is_current", F.lit(True))
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(TARGET_TABLE)
    )
    print("Silver Products table created successfully (SCD Type 2 initialized).")

else:
    target = DeltaTable.forName(spark, TARGET_TABLE)

    # Change-detection condition (same business columns as before)
    change_condition = """
        coalesce(target.product_category_name, '') <> coalesce(updates.product_category_name, '')
        OR coalesce(target.product_name_length, -1) <> coalesce(updates.product_name_length, -1)
        OR coalesce(target.product_description_length, -1) <> coalesce(updates.product_description_length, -1)
        OR coalesce(target.product_photos_qty, -1) <> coalesce(updates.product_photos_qty, -1)
        OR coalesce(target.product_weight_g, -1) <> coalesce(updates.product_weight_g, -1)
        OR coalesce(target.product_length_cm, -1) <> coalesce(updates.product_length_cm, -1)
        OR coalesce(target.product_height_cm, -1) <> coalesce(updates.product_height_cm, -1)
        OR coalesce(target.product_width_cm, -1) <> coalesce(updates.product_width_cm, -1)
    """

    # Step 1: Find records that need to be expired (matched + changed)
    # Step 2: Combine with ALL incoming records (which need to be inserted as new "current" rows)
    staged_updates = (
        valid_products.alias("updates")
        .join(
            target.toDF().filter("is_current = true").alias("target"),
            "product_id"
        )
        .where(change_condition)
        .selectExpr("NULL as mergeKey", "updates.*")   # marks these for expiry only
        .union(
            valid_products.selectExpr("product_id as mergeKey", "*")  # all rows for insert
        )
    )

    (
        target.alias("target")
        .merge(
            staged_updates.alias("staged"),
            "target.product_id = staged.mergeKey AND target.is_current = true"
        )
        .whenMatchedUpdate(
            condition=change_condition.replace("updates.", "staged."),
            set={
                "is_current": "false",
                "effective_end_date": "current_date()"
            }
        )
        .whenNotMatchedInsert(
    values={
        "product_id": "staged.product_id",
        "product_category_name": "staged.product_category_name",
        "product_name_length": "staged.product_name_length",
        "product_description_length": "staged.product_description_length",
        "product_photos_qty": "staged.product_photos_qty",
        "product_weight_g": "staged.product_weight_g",
        "product_length_cm": "staged.product_length_cm",
        "product_height_cm": "staged.product_height_cm",
        "product_width_cm": "staged.product_width_cm",
        "product_volume_cm3": "staged.product_volume_cm3",
        "ingestion_timestamp": "staged.ingestion_timestamp",
        "ingestion_date": "staged.ingestion_date",
        "pipeline_name": "staged.pipeline_name",
        "run_id": "staged.run_id",
        "effective_start_date": "CAST(current_date() AS DATE)",
        "effective_end_date": "CAST(NULL AS DATE)",
        "is_current": "CAST(true AS BOOLEAN)"
    }
)
        .execute()
    )

    print("Silver Products table merged successfully (SCD Type 2).")

Silver Products table merged successfully (SCD Type 2).


In [0]:
# Validation Summary
print("Silver Products Validation Summary")
print(f"Source Records              : {total_rows}")
print(f"Primary Key Validation      : {primary_key_status}")
print(f"Duplicate Records           : {duplicate_rows}")
print(f"Business Rule Violations    : {invalid_products.count()}")
print(f"Quarantined Records         : {invalid_products.count()}")
print(f"Valid Records Loaded        : {valid_products.count()}")
print(f"Target Silver Records       : {spark.table(TARGET_TABLE).count()}")

Silver Products Validation Summary
Source Records              : 3000
Primary Key Validation      : True
Duplicate Records           : 0
Business Rule Violations    : 0
Quarantined Records         : 0
Valid Records Loaded        : 3000
Target Silver Records       : 3000


# Engineering Observations

- Product data was standardized by trimming and converting category names to lowercase.
- Numeric attributes such as weight, dimensions, description length, and photo count were converted from string to integer.
- Primary key validation confirmed all product_id values were unique.
- No duplicate records were detected.
- Business rule validation confirmed all products had valid categories and positive physical dimensions.
- No records were quarantined.
- Data was loaded into the Silver layer using Delta Lake MERGE, implementing SCD Type 2 semantics.